In [1]:
!nvidia-smi

Mon Dec  8 06:53:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Setup & Imports

In [2]:
import os
import glob
import json
import numpy as np
import pandas as pd
from collections import defaultdict

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

In [3]:
tf.random.set_seed(42)
np.random.seed(42)

# Mount Drive & Define Paths

In [4]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


In [5]:
ROOT_DIR = "/users/"
DATA_ROOT = os.path.join(ROOT_DIR, "preprocessed-dataset")
MODEL_ROOT = os.path.join(ROOT_DIR, "Models")
FED_PERSONALIZED_DIR = os.path.join(MODEL_ROOT, "FedPersonalized")

os.makedirs(FED_PERSONALIZED_DIR, exist_ok=True)
FED_PERSONALIZED_DIR

'/users/'

# Auto-find Dataset Paths

In [6]:
def auto_find_dataset(folder_prefix: str) -> str:
    matches = glob.glob(f"{DATA_ROOT}/{folder_prefix}*")
    if len(matches) == 0:
        raise ValueError(f"No folder found for prefix: {folder_prefix}")
    return matches[0]

DATASET_PATHS = {
    "CSE_CIC_IDS2018": auto_find_dataset("CSE_CIC_IDS2018"),
    "CIC_IIoT_2025":   auto_find_dataset("CIC_IIoT_2025"),
    "CIC_BCCC_IoMT_2024": auto_find_dataset("CIC_BCCC_NRC_IoMT_2024"),
    "Combined": auto_find_dataset("Combined"),
}

print("Resolved dataset paths:")
for k, v in DATASET_PATHS.items():
    print(f"  {k}: {v}")

Resolved dataset paths:
  CSE_CIC_IDS2018: /users/
  CIC_IIoT_2025: /users/
  CIC_BCCC_IoMT_2024: /users/
  Combined: /users/


# Data Loader Utility

In [7]:
def load_preprocessed_dataset(ds_name):
    path = DATASET_PATHS[ds_name]

    # Case 1: Standard datasets (CIC_IIoT_2025, CSE_CIC_IDS2018, IoMT_2024)
    latent_files = {
        "X_train": "train_latent.npy",
        "X_val":   "val_latent.npy",
        "X_test":  "test_latent.npy",
        "y_train": "y_train.npy",
        "y_val":   "y_val.npy",
        "y_test":  "y_test.npy",
    }

    # Case 2: Combined dataset uses "combined_*" prefix
    if ds_name == "Combined":
        latent_files = {
            "X_train": "combined_train_latent.npy",
            "X_val":   "combined_val_latent.npy",
            "X_test":  "combined_test_latent.npy",
            "y_train": "combined_y_train.npy",
            "y_val":   "combined_y_val.npy",
            "y_test":  "combined_y_test.npy",
        }

    # Load all arrays
    X_train = np.load(os.path.join(path, latent_files["X_train"]))
    X_val   = np.load(os.path.join(path, latent_files["X_val"]))
    X_test  = np.load(os.path.join(path, latent_files["X_test"]))

    y_train = np.load(os.path.join(path, latent_files["y_train"]))
    y_val   = np.load(os.path.join(path, latent_files["y_val"]))
    y_test  = np.load(os.path.join(path, latent_files["y_test"]))

    num_classes = len(np.unique(y_train))

    print(f"""📂 Loaded dataset: {ds_name}
  Path: {path}
  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}
  y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}
  Num classes: {num_classes}
""")

    return X_train, y_train, X_val, y_val, X_test, y_test, num_classes



def make_federated_clients(
    X_train: np.ndarray,
    y_train: np.ndarray,
    num_clients: int = 10,
    shuffle: bool = True,
):
    n_samples = len(X_train)
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(SEED)
        rng.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    base_size = n_samples // num_clients
    remainder = n_samples % num_clients

    client_splits = []
    start = 0
    for i in range(num_clients):
        size = base_size + (1 if i < remainder else 0)
        end = start + size
        X_c = X_train[start:end]
        y_c = y_train[start:end]
        client_splits.append((X_c, y_c))
        print(f"  -> Client {i}: {len(X_c)} samples")
        start = end

    return client_splits

# Federated Client Splitter

In [8]:
def make_federated_clients(X, y, num_clients: int = 10):
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    np.random.shuffle(indices)

    splits = np.array_split(indices, num_clients)
    client_data = {}
    for cid, idx in enumerate(splits):
        X_c = X[idx]
        y_c = y[idx]
        client_data[cid] = (X_c, y_c)
        print(f"  -> Client {cid}: {X_c.shape[0]} samples")

    return client_data

# FALCON-ID Local Model

In [9]:
def build_falcon_id_local_model(
    input_dim,
    num_classes,
    learning_rate=1e-3,
    l2_reg=1e-4,
    dropout_rate=0.4,
):
    inp = layers.Input(shape=(input_dim,), name="latent_input")
    x = layers.Reshape((input_dim, 1))(inp)  # (batch, 64, 1)

    # ---- Residual Block 1 ----
    shortcut = layers.Conv1D(
        64, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation("relu")(x1)
    x1 = layers.Conv1D(
        64, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x1)
    x1 = layers.BatchNormalization()(x1)

    x = layers.Add()([shortcut, x1])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # ---- Residual Block 2 ----
    shortcut2 = layers.Conv1D(
        128, 1, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)

    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.Activation("relu")(x2)
    x2 = layers.Conv1D(
        128, 3, padding="same",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x2)
    x2 = layers.BatchNormalization()(x2)

    x = layers.Add()([shortcut2, x2])
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.SpatialDropout1D(0.2)(x)

    # ---- BiLSTM ----
    x = layers.Bidirectional(
        layers.LSTM(
            128,
            return_sequences=True,
            kernel_regularizer=regularizers.l2(l2_reg),
        )
    )(x)

    # ---- Additive Attention (note: time axis has small length after pooling) ----
    score = layers.Dense(128, activation="tanh")(x)
    attn_weights = layers.Dense(1, activation="softmax")(score)
    context = layers.Lambda(lambda z: tf.reduce_sum(z[0] * z[1], axis=1))(
        [x, attn_weights]
    )

    # ---- Classifier ----
    x = layers.Dense(
        256, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(context)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(l2_reg)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    out = layers.Dense(num_classes, activation="softmax", name="logits")(x)

    model = models.Model(inputs=inp, outputs=out, name="FALCON_ID_CNN_BiLSTM_Attn")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model

tmp_X_train, tmp_y_train, _, _, _, _, tmp_num_classes = load_preprocessed_dataset("CIC_IIoT_2025")
tmp_model = build_falcon_id_local_model(input_dim=tmp_X_train.shape[1], num_classes=tmp_num_classes)
tmp_model.summary()
del tmp_X_train, tmp_y_train, tmp_model

📂 Loaded dataset: CIC_IIoT_2025
  Path: /users/
  X_train: (29424, 64), X_val: (6305, 64), X_test: (6306, 64)
  y_train: (29424,), y_val: (6305,), y_test: (6306,)
  Num classes: 7



Model: "FALCON_ID_CNN_BiLSTM_Attn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ latent_input        │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 64, 1)     │          0 │ latent_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 64, 64)    │        256 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 64, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 64, 64)    │        128 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 64)    │          0 │ conv1d[0][0],     │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 32, 64)    │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 32, 128)   │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 128)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 32, 128)   │     49,280 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 32, 128)   │      8,320 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 128)   │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 128)   │          0 │ conv1d_3[0][0],   │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 128)   │          0 │ add_1[0][0]       │
│ (Activation)        │                   │            │                 

 Total params: 493,896 (1.88 MB)

 Trainable params: 492,360 (1.88 MB)

 Non-trainable params: 1,536 (6.00 KB)

# Helper Functions: Weights & FedAvg

In [10]:
def get_model_weights(model):
    return model.get_weights()

def set_model_weights(model, weights):
    model.set_weights(weights)

def fed_avg(client_weights, client_sizes):
    num_clients = len(client_weights)
    total_samples = np.sum(client_sizes)

    avg_weights = []
    for layer_idx in range(len(client_weights[0])):
        layer_sum = np.zeros_like(client_weights[0][layer_idx])
        for ck, nk in zip(client_weights, client_sizes):
            layer_sum += (nk / total_samples) * ck[layer_idx]
        avg_weights.append(layer_sum)
    return avg_weights

def evaluate_global_model(model, X_train, y_train, X_val, y_val, X_test, y_test, batch_size=4096):
    train_loss, train_acc = model.evaluate(X_train, y_train, batch_size=batch_size, verbose=0)
    val_loss, val_acc     = model.evaluate(X_val,   y_val,   batch_size=batch_size, verbose=0)
    test_loss, test_acc   = model.evaluate(X_test,  y_test,  batch_size=batch_size, verbose=0)
    return (train_loss, train_acc, val_loss, val_acc, test_loss, test_acc)

# FedProx Training for One Dataset

In [11]:
def run_fedprox_for_dataset(
    dataset_name: str,
    num_clients: int = 10,
    num_rounds: int = 10,
    local_epochs: int = 1,
    batch_size: int = 512,
    mu: float = 0.001,
    base_lr: float = 1e-3,
):
    print("\n" + "#" * 80)
    print(f"### 🌐 FedProx for Dataset: {dataset_name}")
    print("#" * 80)

    # 1) Load dataset
    X_train, y_train, X_val, y_val, X_test, y_test, num_classes = load_preprocessed_dataset(dataset_name)
    input_dim = X_train.shape[1]

    # 2) Make federated clients
    print(f"\n👥 Creating {num_clients} federated clients for {dataset_name}...")
    client_splits = make_federated_clients(X_train, y_train, num_clients=num_clients)

    # 3) Init global model
    print(f"\n🧱 Building global model for {dataset_name} (FedProx)...")
    global_model = build_falcon_id_local_model(input_dim=input_dim, num_classes=num_classes, learning_rate=base_lr)

    history = {
        "round": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    for r in range(1, num_rounds + 1):
        print(f"\n===== 🔄 FedProx Round {r}/{num_rounds} — Dataset: {dataset_name} =====")

        global_weights = get_model_weights(global_model)
        client_weights = []
        client_sizes = []

        # 4) Local training on each client
        for cid, (X_c, y_c) in client_splits.items():
            print(f"  -> Client {cid}: local training on {X_c.shape[0]} samples...")
            local_model = build_falcon_id_local_model(
                input_dim=input_dim,
                num_classes=num_classes,
                learning_rate=base_lr,
            )
            set_model_weights(local_model, global_weights)

            local_model.fit(
                X_c,
                y_c,
                epochs=local_epochs,
                batch_size=batch_size,
                verbose=0,
                shuffle=True,
            )

            w_i = get_model_weights(local_model)

            # ---- FedProx shrink step: w_i_tilde = w_i - mu (w_i - w_global) ----
            w_i_prox = []
            for w_layer, w_g_layer in zip(w_i, global_weights):
                w_i_prox.append(w_layer - mu * (w_layer - w_g_layer))

            client_weights.append(w_i_prox)
            client_sizes.append(X_c.shape[0])

            tf.keras.backend.clear_session()
            del local_model

        # 5) Aggregate
        new_global_weights = fed_avg(client_weights, client_sizes)
        set_model_weights(global_model, new_global_weights)

        # 6) Evaluate global model
        train_loss, train_acc, val_loss, val_acc, test_loss, test_acc = \
            evaluate_global_model(global_model, X_train, y_train, X_val, y_val, X_test, y_test)

        print(f"  -> Train | loss: {train_loss:.4f}, acc: {train_acc:.4f}")
        print(f"  -> Val   | loss: {val_loss:.4f}, acc: {val_acc:.4f}")
        print(f"  -> Test  | loss: {test_loss:.4f}, acc: {test_acc:.4f}")

        history["round"].append(r)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

    print(f"\n🎯 Final FedProx Test Performance on {dataset_name}:")
    print(f"   Test loss: {history['test_loss'][-1]:.4f}, Test acc: {history['test_acc'][-1]:.4f}")

    return global_model, history, client_splits

# Local Fine-Tuning (Personalization) per Client

In [12]:
def run_local_finetuning(
    dataset_name: str,
    global_model,
    client_splits,
    num_epochs: int = 1,
    batch_size: int = 256,
    base_lr: float = 5e-4,
):
    print(f"\n🔧 Local Fine-Tuning (Personalization) — Dataset: {dataset_name}")
    global_weights = get_model_weights(global_model)

    rows = []

    for cid, (X_c, y_c) in client_splits.items():
        print(f"\n  -> Client {cid}: Fine-tuning on {X_c.shape[0]} samples...")
        input_dim = X_c.shape[1]
        num_classes = global_model.output_shape[-1]

        local_model = build_falcon_id_local_model(
            input_dim=input_dim,
            num_classes=num_classes,
            learning_rate=base_lr,
        )
        set_model_weights(local_model, global_weights)

        # Fine-tune
        history = local_model.fit(
            X_c,
            y_c,
            epochs=num_epochs,
            batch_size=batch_size,
            verbose=0,
            shuffle=True,
        )

        loss, acc = local_model.evaluate(X_c, y_c, batch_size=batch_size, verbose=0)
        print(f"     Personalized client-{cid} accuracy (on its own data): {acc:.4f}")

        rows.append({
            "dataset": dataset_name,
            "client_id": cid,
            "num_samples": int(X_c.shape[0]),
            "personalized_loss": float(loss),
            "personalized_acc": float(acc),
        })

        tf.keras.backend.clear_session()
        del local_model

    df_clients = pd.DataFrame(rows)
    return df_clients

# Run FedProx + Local Fine-Tuning on ALL Datasets

In [13]:
NUM_CLIENTS = 10
FEDPROX_ROUNDS = 5
FEDPROX_LOCAL_EPOCHS = 1
FEDPROX_MU = 0.001
FEDPROX_LR = 1e-3

LOCAL_FT_EPOCHS = 1
LOCAL_FT_LR = 5e-4

all_global_summaries = []
all_client_personal_summaries = []

for ds_name in DATASET_PATHS.keys():
    # 1) FedProx
    fedprox_model, fedprox_history, client_splits = run_fedprox_for_dataset(
        dataset_name=ds_name,
        num_clients=NUM_CLIENTS,
        num_rounds=FEDPROX_ROUNDS,
        local_epochs=FEDPROX_LOCAL_EPOCHS,
        batch_size=512,
        mu=FEDPROX_MU,
        base_lr=FEDPROX_LR,
    )

    # Final round metrics
    final_test_loss = fedprox_history["test_loss"][-1]
    final_test_acc  = fedprox_history["test_acc"][-1]

    all_global_summaries.append({
        "dataset": ds_name,
        "method": "FedProx",
        "num_clients": NUM_CLIENTS,
        "num_rounds": FEDPROX_ROUNDS,
        "mu": FEDPROX_MU,
        "final_test_loss": final_test_loss,
        "final_test_acc": final_test_acc,
    })

    # 2) Local Fine-Tuning (Personalization)
    df_clients = run_local_finetuning(
        dataset_name=ds_name,
        global_model=fedprox_model,
        client_splits=client_splits,
        num_epochs=LOCAL_FT_EPOCHS,
        batch_size=256,
        base_lr=LOCAL_FT_LR,
    )

    all_client_personal_summaries.append(df_clients)

    # 3) Save model weights for this dataset
    ds_model_dir = os.path.join(FED_PERSONALIZED_DIR, ds_name)
    os.makedirs(ds_model_dir, exist_ok=True)

    fedprox_model_path = os.path.join(ds_model_dir, f"{ds_name}_FedProx_global.h5")
    fedprox_model.save(fedprox_model_path)
    print(f"\n💾 Saved FedProx global model for {ds_name} at: {fedprox_model_path}")

    # 4) Save FedProx training history
    hist_df = pd.DataFrame(fedprox_history)
    hist_csv_path = os.path.join(ds_model_dir, f"{ds_name}_FedProx_history.csv")
    hist_df.to_csv(hist_csv_path, index=False)
    print(f"💾 Saved FedProx round-wise history for {ds_name} at: {hist_csv_path}")


################################################################################
### 🌐 FedProx for Dataset: CSE_CIC_IDS2018
################################################################################
📂 Loaded dataset: CSE_CIC_IDS2018
  Path: /users/
  X_train: (6737603, 64), X_val: (1443772, 64), X_test: (1443773, 64)
  y_train: (6737603,), y_val: (1443772,), y_test: (1443773,)
  Num classes: 15


👥 Creating 10 federated clients for CSE_CIC_IDS2018...
  -> Client 0: 673761 samples
  -> Client 1: 673761 samples
  -> Client 2: 673761 samples
  -> Client 3: 673760 samples
  -> Client 4: 673760 samples
  -> Client 5: 673760 samples
  -> Client 6: 673760 samples
  -> Client 7: 673760 samples
  -> Client 8: 673760 samples
  -> Client 9: 673760 samples

🧱 Building global model for CSE_CIC_IDS2018 (FedProx)...

===== 🔄 FedProx Round 1/5 — Dataset: CSE_CIC_IDS2018 =====
  -> Client 0: local training on 673761 samples...


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 16, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


  -> Client 1: local training on 673761 samples...
  -> Client 2: local training on 673761 samples...
  -> Client 3: local training on 673760 samples...
  -> Client 4: local training on 673760 samples...
  -> Client 5: local training on 673760 samples...
  -> Client 6: local training on 673760 samples...
  -> Client 7: local training on 673760 samples...
  -> Client 8: local training on 673760 samples...
  -> Client 9: local training on 673760 samples...
  -> Train | loss: 0.4074, acc: 0.9251
  -> Val   | loss: 0.4078, acc: 0.9251
  -> Test  | loss: 0.4073, acc: 0.9251

===== 🔄 FedProx Round 2/5 — Dataset: CSE_CIC_IDS2018 =====
  -> Client 0: local training on 673761 samples...
  -> Client 1: local training on 673761 samples...
  -> Client 2: local training on 673761 samples...
  -> Client 3: local training on 673760 samples...
  -> Client 4: local training on 673760 samples...
  -> Client 5: local training on 673760 samples...
  -> Client 6: local training on 673760 samples...
  -> Cl


💾 Saved FedProx global model for CSE_CIC_IDS2018 at: /users/
💾 Saved FedProx round-wise history for CSE_CIC_IDS2018 at: /users/

################################################################################
### 🌐 FedProx for Dataset: CIC_IIoT_2025
################################################################################
📂 Loaded dataset: CIC_IIoT_2025
  Path: /users/
  X_train: (29424, 64), X_val: (6305, 64), X_test: (6306, 64)
  y_train: (29424,), y_val: (6305,), y_test: (6306,)
  Num classes: 7


👥 Creating 10 federated clients for CIC_IIoT_2025...
  -> Client 0: 2943 samples
  -> Client 1: 2943 samples
  -> Client 2: 2943 samples
  -> Client 3: 2943 samples
  -> Client 4: 2942 samples
  -> Client 5: 2942 samples
  -> Client 6: 2942 samples
  -> Client 7: 2942 samples
  -> Client 8: 2942 samples
  -> Client 9: 2942 samples

🧱 Building global model for CIC_IIoT_2025 (FedProx)...

===== 🔄 FedProx Round 1/5 — Dataset: CIC_IIoT_2025 =====
  -> Client 0: local training on 2943 


💾 Saved FedProx global model for CIC_IIoT_2025 at: /users/
💾 Saved FedProx round-wise history for CIC_IIoT_2025 at: /users/

################################################################################
### 🌐 FedProx for Dataset: CIC_BCCC_IoMT_2024
################################################################################
📂 Loaded dataset: CIC_BCCC_IoMT_2024
  Path: /users/
  X_train: (2369719, 64), X_val: (507797, 64), X_test: (507797, 64)
  y_train: (2369719,), y_val: (507797,), y_test: (507797,)
  Num classes: 15


👥 Creating 10 federated clients for CIC_BCCC_IoMT_2024...
  -> Client 0: 236972 samples
  -> Client 1: 236972 samples
  -> Client 2: 236972 samples
  -> Client 3: 236972 samples
  -> Client 4: 236972 samples
  -> Client 5: 236972 samples
  -> Client 6: 236972 samples
  -> Client 7: 236972 samples
  -> Client 8: 236972 samples
  -> Client 9: 236971 samples

🧱 Building global model for CIC_BCCC_IoMT_2024 (FedProx)...

===== 🔄 FedProx Round 1/5 — Dataset: CIC_BCCC_


💾 Saved FedProx global model for CIC_BCCC_IoMT_2024 at: /users/
💾 Saved FedProx round-wise history for CIC_BCCC_IoMT_2024 at: /users/

################################################################################
### 🌐 FedProx for Dataset: Combined
################################################################################
📂 Loaded dataset: Combined
  Path: /users/
  X_train: (9136746, 64), X_val: (1957874, 64), X_test: (1957876, 64)
  y_train: (9136746,), y_val: (1957874,), y_test: (1957876,)
  Num classes: 37


👥 Creating 10 federated clients for Combined...
  -> Client 0: 913675 samples
  -> Client 1: 913675 samples
  -> Client 2: 913675 samples
  -> Client 3: 913675 samples
  -> Client 4: 913675 samples
  -> Client 5: 913675 samples
  -> Client 6: 913674 samples
  -> Client 7: 913674 samples
  -> Client 8: 913674 samples
  -> Client 9: 913674 samples

🧱 Building global model for Combined (FedProx)...

===== 🔄 FedProx Round 1/5 — Dataset: Combined =====
  -> Client 0: local


💾 Saved FedProx global model for Combined at: /users/
💾 Saved FedProx round-wise history for Combined at: /users/


# Save Global & Personalized Client Summaries

In [14]:
df_global = pd.DataFrame(all_global_summaries)
global_csv_path = os.path.join(FED_PERSONALIZED_DIR, "FedProx_global_summary_all_datasets.csv")
df_global.to_csv(global_csv_path, index=False)
print("\n📊 Global FedProx summary saved at:", global_csv_path)
display(df_global)


📊 Global FedProx summary saved at: /users/


,dataset,method,num_clients,num_rounds,mu,final_test_loss,final_test_acc
0,CSE_CIC_IDS2018,FedProx,10,5,0.001,0.120658,0.970247
1,CIC_IIoT_2025,FedProx,10,5,0.001,1.775854,0.351570
2,CIC_BCCC_IoMT_2024,FedProx,10,5,0.001,0.122953,0.966431
3,Combined,FedProx,10,5,0.001,0.119344,0.967914


In [15]:
df_clients_all = pd.concat(all_client_personal_summaries, axis=0).reset_index(drop=True)
clients_csv_path = os.path.join(FED_PERSONALIZED_DIR, "FedProx_local_personalization_clients.csv")
df_clients_all.to_csv(clients_csv_path, index=False)
print("\n📊 Local personalization (per-client) summary saved at:", clients_csv_path)
display(df_clients_all.head())


📊 Local personalization (per-client) summary saved at: /users/


,dataset,client_id,num_samples,personalized_loss,personalized_acc
0,CSE_CIC_IDS2018,0,673761,0.105825,0.971002
1,CSE_CIC_IDS2018,1,673761,0.107117,0.971006
2,CSE_CIC_IDS2018,2,673761,0.104275,0.971558
3,CSE_CIC_IDS2018,3,673760,0.110034,0.969501
4,CSE_CIC_IDS2018,4,673760,0.104393,0.971120


# Quick Comparison Stub with Previous Methods

In [16]:
df_global["method_type"] = "Personalized_FL_FedProx"

comparison_stub_path = os.path.join(FED_PERSONALIZED_DIR, "FedProx_global_comparison_stub.csv")
df_global.to_csv(comparison_stub_path, index=False)
print("\n📊 Comparison stub saved at:", comparison_stub_path)
df_global


📊 Comparison stub saved at: /users/


,dataset,method,num_clients,num_rounds,mu,final_test_loss,final_test_acc,method_type
0,CSE_CIC_IDS2018,FedProx,10,5,0.001,0.120658,0.970247,Personalized_FL_FedProx
1,CIC_IIoT_2025,FedProx,10,5,0.001,1.775854,0.351570,Personalized_FL_FedProx
2,CIC_BCCC_IoMT_2024,FedProx,10,5,0.001,0.122953,0.966431,Personalized_FL_FedProx
3,Combined,FedProx,10,5,0.001,0.119344,0.967914,Personalized_FL_FedProx
